# DemandWise — Análise Exploratória de Vendas

Esta etapa investiga a evolução da demanda, tendência, sazonalidade e diferenças entre lojas e produtos. As análises usam a base processada pelo pipeline `src/make_dataset.py` e gráficos interativos em Plotly.

## Perguntas de negócio

1. Como as vendas evoluíram ao longo do tempo?
2. Existe tendência de crescimento?
3. Quais meses e dias da semana concentram maior demanda?
4. Quais lojas e produtos vendem mais?
5. Quais produtos apresentam maior variação sazonal?
6. Como as vendas dos principais produtos variam entre lojas?

## 1. Configuração do ambiente

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")
pio.templates.default = "plotly_white"

project_candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next(
    (path for path in project_candidates if (path / "data" / "processed" / "train_processed.csv").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Base processada não encontrada. Execute 'python src/make_dataset.py' a partir da raiz do projeto."
    )

TRAIN_PATH = PROJECT_ROOT / "data" / "processed" / "train_processed.csv"

COLORS = {
    "primary": "#2563EB",
    "primary_light": "#93C5FD",
    "secondary": "#0F766E",
    "accent": "#F59E0B",
    "muted": "#94A3B8",
    "grid": "#E5E7EB",
    "text": "#1F2937",
}

MONTH_NAMES = {
    1: "Jan", 2: "Fev", 3: "Mar", 4: "Abr",
    5: "Mai", 6: "Jun", 7: "Jul", 8: "Ago",
    9: "Set", 10: "Out", 11: "Nov", 12: "Dez",
}
WEEKDAY_NAMES = {
    0: "Segunda", 1: "Terça", 2: "Quarta", 3: "Quinta",
    4: "Sexta", 5: "Sábado", 6: "Domingo",
}

def format_units(value):
    """Formata unidades para rótulos compactos em português."""
    if abs(value) >= 1_000_000:
        return f"{value / 1_000_000:.1f} mi".replace(".", ",")
    if abs(value) >= 1_000:
        return f"{value / 1_000:.1f} mil".replace(".", ",")
    return f"{value:,.0f}".replace(",", ".")

def style_figure(fig, title, height=480, show_legend=False):
    """Aplica identidade visual consistente aos gráficos."""
    fig.update_layout(
        title={"text": title, "x": 0.01, "xanchor": "left"},
        height=height,
        font={"family": "Arial, sans-serif", "size": 13, "color": COLORS["text"]},
        margin={"l": 70, "r": 35, "t": 80, "b": 65},
        hoverlabel={"font_size": 13},
        hovermode="closest",
        showlegend=show_legend,
        separators=",.",
        paper_bgcolor="white",
        plot_bgcolor="white",
    )
    fig.update_xaxes(showgrid=False, automargin=True, title_standoff=12)
    fig.update_yaxes(gridcolor=COLORS["grid"], zeroline=False, automargin=True, title_standoff=12)
    return fig

print(f"Raiz do projeto: {PROJECT_ROOT.resolve()}")

## 2. Carregamento e validação da base

In [ ]:
train = pd.read_csv(TRAIN_PATH, parse_dates=["date"])

required_columns = {
    "date", "store", "item", "sales", "year", "month",
    "day", "day_of_week", "week_of_year", "quarter", "is_weekend",
}
missing_columns = required_columns - set(train.columns)
if missing_columns:
    raise ValueError(f"Colunas esperadas ausentes: {sorted(missing_columns)}")

summary = pd.DataFrame(
    {
        "Indicador": [
            "Registros", "Data inicial", "Data final", "Lojas",
            "Produtos", "Vendas totais", "Valores ausentes",
        ],
        "Valor": [
            f"{len(train):,}",
            train["date"].min().strftime("%d/%m/%Y"),
            train["date"].max().strftime("%d/%m/%Y"),
            train["store"].nunique(),
            train["item"].nunique(),
            format_units(train["sales"].sum()),
            int(train.isna().sum().sum()),
        ],
    }
)
display(summary)

## 3. Preparação das agregações

Os gráficos usam dados agregados para manter a análise rápida e legível. Comparações sazonais usam **média diária**, evitando favorecer meses com mais dias.

In [ ]:
daily_sales = train.groupby("date", as_index=False)["sales"].sum()
daily_sales["moving_average_28"] = daily_sales["sales"].rolling(28, min_periods=1).mean()
daily_sales["year"] = daily_sales["date"].dt.year
daily_sales["month"] = daily_sales["date"].dt.month
daily_sales["day_of_week"] = daily_sales["date"].dt.dayofweek
daily_sales["is_weekend"] = daily_sales["day_of_week"].isin([5, 6])

monthly_sales = (
    train.set_index("date")["sales"]
    .resample("MS")
    .sum()
    .rename("sales")
    .reset_index()
)
monthly_sales["moving_average_3"] = monthly_sales["sales"].rolling(3, min_periods=1).mean()

yearly_sales = train.groupby("year", as_index=False)["sales"].sum()
yearly_sales["growth_pct"] = yearly_sales["sales"].pct_change() * 100

monthly_profile = daily_sales.groupby("month", as_index=False)["sales"].mean()
monthly_profile["month_name"] = monthly_profile["month"].map(MONTH_NAMES)
monthly_profile["seasonal_index"] = monthly_profile["sales"] / daily_sales["sales"].mean() * 100

monthly_by_year = daily_sales.groupby(["year", "month"], as_index=False)["sales"].mean()
monthly_by_year["month_name"] = monthly_by_year["month"].map(MONTH_NAMES)

weekday_profile = daily_sales.groupby("day_of_week", as_index=False)["sales"].mean()
weekday_profile["day_name"] = weekday_profile["day_of_week"].map(WEEKDAY_NAMES)

store_sales = train.groupby("store", as_index=False)["sales"].sum().sort_values("sales", ascending=False)
item_sales = train.groupby("item", as_index=False)["sales"].sum().sort_values("sales", ascending=False)
top_10_items = item_sales.head(10).copy()
top_10_stores = store_sales.head(10).copy()

print("Agregações preparadas com sucesso.")

## 4. Evolução diária e tendência geral

A série diária revela oscilações de curto prazo. A média móvel de 28 dias suaviza o ruído e torna a direção geral da demanda mais visível.

In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=daily_sales["date"], y=daily_sales["sales"],
        mode="lines", name="Vendas diárias",
        line={"color": COLORS["primary_light"], "width": 1},
        opacity=0.55,
        hovertemplate="%{x|%d/%m/%Y}<br>Vendas: %{y:,.0f}<extra></extra>",
    )
)
fig.add_trace(
    go.Scatter(
        x=daily_sales["date"], y=daily_sales["moving_average_28"],
        mode="lines", name="Média móvel — 28 dias",
        line={"color": COLORS["primary"], "width": 3},
        hovertemplate="%{x|%d/%m/%Y}<br>Média 28d: %{y:,.0f}<extra></extra>",
    )
)
style_figure(fig, "Evolução diária das vendas e tendência de 28 dias", height=520, show_legend=True)
fig.update_layout(hovermode="x unified", legend={"orientation": "h", "y": 1.08, "x": 0})
fig.update_xaxes(title="Data", rangeslider={"visible": True, "thickness": 0.08})
fig.update_yaxes(title="Unidades vendidas por dia", rangemode="tozero")
fig.show()

## 5. Vendas por ano

O volume anual permite avaliar crescimento estrutural e comparar a expansão entre anos completos.

In [ ]:
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(
    go.Bar(
        x=yearly_sales["year"], y=yearly_sales["sales"] / 1_000_000,
        name="Vendas", marker_color=COLORS["primary"],
        text=[format_units(value) for value in yearly_sales["sales"]],
        textposition="outside", cliponaxis=False,
        hovertemplate="Ano: %{x}<br>Vendas: %{y:.2f} milhões<extra></extra>",
    ),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(
        x=yearly_sales["year"], y=yearly_sales["growth_pct"],
        name="Crescimento anual", mode="lines+markers+text",
        line={"color": COLORS["accent"], "width": 3},
        marker={"size": 9},
        text=["" if pd.isna(value) else f"{value:.1f}%" for value in yearly_sales["growth_pct"]],
        textposition="top center",
        hovertemplate="Ano: %{x}<br>Crescimento: %{y:.1f}%<extra></extra>",
    ),
    secondary_y=True,
)
style_figure(fig, "Vendas anuais e crescimento em relação ao ano anterior", height=500, show_legend=True)
fig.update_layout(legend={"orientation": "h", "y": 1.10, "x": 0}, bargap=0.35)
fig.update_xaxes(title="Ano", dtick=1)
fig.update_yaxes(title="Vendas (milhões de unidades)", secondary_y=False, rangemode="tozero")
fig.update_yaxes(title="Crescimento anual (%)", secondary_y=True, showgrid=False)
fig.show()

## 6. Vendas por mês ao longo do tempo

A agregação mensal reduz a volatilidade diária e evidencia ciclos recorrentes e mudanças de nível na demanda.

In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=monthly_sales["date"], y=monthly_sales["sales"],
        name="Vendas mensais", marker_color=COLORS["primary_light"],
        hovertemplate="%{x|%b/%Y}<br>Vendas: %{y:,.0f}<extra></extra>",
    )
)
fig.add_trace(
    go.Scatter(
        x=monthly_sales["date"], y=monthly_sales["moving_average_3"],
        name="Média móvel — 3 meses", mode="lines",
        line={"color": COLORS["primary"], "width": 3},
        hovertemplate="%{x|%b/%Y}<br>Média 3m: %{y:,.0f}<extra></extra>",
    )
)
style_figure(fig, "Demanda mensal e tendência de curto prazo", height=500, show_legend=True)
fig.update_layout(hovermode="x unified", legend={"orientation": "h", "y": 1.08, "x": 0})
fig.update_xaxes(title="Mês")
fig.update_yaxes(title="Unidades vendidas", rangemode="tozero")
fig.show()

## 7. Sazonalidade mensal

O perfil abaixo compara a média diária de cada mês. O índice sazonal usa 100 como demanda média: valores acima de 100 indicam meses mais fortes que a média geral.

In [ ]:
bar_colors = [COLORS["accent"] if value == monthly_profile["sales"].max() else COLORS["primary"] for value in monthly_profile["sales"]]
fig = go.Figure(
    go.Bar(
        x=monthly_profile["month_name"], y=monthly_profile["sales"],
        marker_color=bar_colors,
        customdata=monthly_profile[["seasonal_index"]],
        text=[format_units(value) for value in monthly_profile["sales"]],
        textposition="outside", cliponaxis=False,
        hovertemplate="%{x}<br>Média diária: %{y:,.0f}<br>Índice sazonal: %{customdata[0]:.1f}<extra></extra>",
    )
)
style_figure(fig, "Média diária de vendas por mês do ano", height=490)
fig.add_hline(
    y=daily_sales["sales"].mean(), line_dash="dot", line_color=COLORS["muted"],
    annotation_text="Média geral", annotation_position="top left",
)
fig.update_xaxes(title="Mês", categoryorder="array", categoryarray=list(MONTH_NAMES.values()))
fig.update_yaxes(title="Média diária de unidades vendidas", rangemode="tozero")
fig.show()

In [ ]:
fig = px.line(
    monthly_by_year, x="month_name", y="sales", color="year", markers=True,
    labels={"month_name": "Mês", "sales": "Média diária de vendas", "year": "Ano"},
    color_discrete_sequence=px.colors.sequential.Blues[3:],
)
style_figure(fig, "Repetição do padrão mensal em cada ano", height=500, show_legend=True)
fig.update_layout(hovermode="x unified", legend={"orientation": "h", "y": 1.08, "x": 0})
fig.update_traces(line={"width": 2.5}, hovertemplate="Ano %{fullData.name}<br>%{x}: %{y:,.0f}<extra></extra>")
fig.update_xaxes(categoryorder="array", categoryarray=list(MONTH_NAMES.values()))
fig.update_yaxes(rangemode="tozero")
fig.show()

## 8. Vendas por dia da semana

A média diária permite comparar os dias em condições equivalentes e identificar picos operacionais recorrentes.

In [ ]:
weekday_colors = [COLORS["accent"] if value == weekday_profile["sales"].max() else COLORS["secondary"] for value in weekday_profile["sales"]]
fig = go.Figure(
    go.Bar(
        x=weekday_profile["day_name"], y=weekday_profile["sales"],
        marker_color=weekday_colors,
        text=[format_units(value) for value in weekday_profile["sales"]],
        textposition="outside", cliponaxis=False,
        hovertemplate="%{x}<br>Média diária: %{y:,.0f}<extra></extra>",
    )
)
style_figure(fig, "Média diária de vendas por dia da semana", height=480)
fig.update_xaxes(title="Dia da semana", categoryorder="array", categoryarray=list(WEEKDAY_NAMES.values()))
fig.update_yaxes(title="Média diária de unidades vendidas", rangemode="tozero")
fig.show()

## 9. Vendas por loja — ranking completo

O dataset possui 10 lojas; portanto, o ranking completo também representa o **Top 10 de lojas por volume**.

In [ ]:
store_ranking = top_10_stores.sort_values("sales", ascending=True)
fig = go.Figure(
    go.Bar(
        x=store_ranking["sales"] / 1_000_000, y=store_ranking["store"].astype(str),
        orientation="h", marker_color=COLORS["primary"],
        text=[format_units(value) for value in store_ranking["sales"]],
        textposition="outside", cliponaxis=False,
        hovertemplate="Loja %{y}<br>Vendas: %{x:.2f} milhões<extra></extra>",
    )
)
style_figure(fig, "Top 10 lojas com maior volume de vendas", height=520)
fig.update_xaxes(title="Vendas (milhões de unidades)", rangemode="tozero")
fig.update_yaxes(title="Loja", type="category")
fig.show()

## 10. Vendas por produto

A visão dos 50 produtos mostra a dispersão de volume do portfólio. Em seguida, o Top 10 isola os itens de maior contribuição.

In [ ]:
items_by_id = item_sales.sort_values("item").copy()
top_item_ids = set(top_10_items["item"])
item_colors = [COLORS["primary"] if item in top_item_ids else COLORS["primary_light"] for item in items_by_id["item"]]
fig = go.Figure(
    go.Bar(
        x=items_by_id["item"], y=items_by_id["sales"] / 1_000_000,
        marker_color=item_colors,
        hovertemplate="Produto %{x}<br>Vendas: %{y:.2f} milhões<extra></extra>",
    )
)
style_figure(fig, "Volume total por produto — Top 10 destacados", height=500)
fig.update_xaxes(title="Produto", dtick=2)
fig.update_yaxes(title="Vendas (milhões de unidades)", rangemode="tozero")
fig.show()

In [ ]:
top_items_ranking = top_10_items.sort_values("sales", ascending=True)
fig = go.Figure(
    go.Bar(
        x=top_items_ranking["sales"] / 1_000_000, y=top_items_ranking["item"].astype(str),
        orientation="h", marker_color=COLORS["secondary"],
        text=[format_units(value) for value in top_items_ranking["sales"]],
        textposition="outside", cliponaxis=False,
        hovertemplate="Produto %{y}<br>Vendas: %{x:.2f} milhões<extra></extra>",
    )
)
style_figure(fig, "Top 10 produtos mais vendidos", height=520)
fig.update_xaxes(title="Vendas (milhões de unidades)", rangemode="tozero")
fig.update_yaxes(title="Produto", type="category")
fig.show()

## 11. Variação entre lojas e produtos

O mapa de calor compara o volume dos 10 produtos mais vendidos em cada loja. Células mais intensas representam maior volume acumulado.

In [ ]:
store_item_sales = train.groupby(["store", "item"], as_index=False)["sales"].sum()
store_top_items = store_item_sales[store_item_sales["item"].isin(top_10_items["item"])]
store_item_matrix = (
    store_top_items.pivot(index="store", columns="item", values="sales")
    .reindex(index=sorted(train["store"].unique()), columns=top_10_items["item"].tolist())
)

fig = go.Figure(
    go.Heatmap(
        z=store_item_matrix.values / 1_000,
        x=[f"Produto {item}" for item in store_item_matrix.columns],
        y=[f"Loja {store}" for store in store_item_matrix.index],
        colorscale=[[0, "#EFF6FF"], [0.5, "#60A5FA"], [1, "#1D4ED8"]],
        colorbar={"title": "Milhares<br>de unidades"},
        hovertemplate="%{y}<br>%{x}<br>Vendas: %{z:,.1f} mil<extra></extra>",
    )
)
style_figure(fig, "Volume dos principais produtos por loja", height=540)
fig.update_xaxes(title="Produto", side="bottom")
fig.update_yaxes(title="Loja", autorange="reversed")
fig.show()

## 12. Produtos com comportamento sazonal mais forte

Para cada produto, o índice sazonal compara sua venda diária média em cada mês com sua própria média anual. O valor 100 representa o nível normal do produto; desvios maiores indicam sazonalidade mais forte.

In [ ]:
daily_item_sales = train.groupby(["date", "item"], as_index=False)["sales"].sum()
daily_item_sales["month"] = daily_item_sales["date"].dt.month
item_month_profile = daily_item_sales.groupby(["item", "month"], as_index=False)["sales"].mean()
item_month_profile["seasonal_index"] = (
    item_month_profile["sales"]
    / item_month_profile.groupby("item")["sales"].transform("mean")
    * 100
)
seasonality_strength = (
    item_month_profile.groupby("item")["seasonal_index"]
    .std()
    .rename("seasonality_strength")
    .sort_values(ascending=False)
    .reset_index()
)
top_seasonal_items = seasonality_strength.head(10)["item"].tolist()
seasonal_matrix = (
    item_month_profile[item_month_profile["item"].isin(top_seasonal_items)]
    .pivot(index="item", columns="month", values="seasonal_index")
    .reindex(index=top_seasonal_items, columns=range(1, 13))
)

fig = go.Figure(
    go.Heatmap(
        z=seasonal_matrix.values,
        x=list(MONTH_NAMES.values()),
        y=[f"Produto {item}" for item in seasonal_matrix.index],
        zmid=100,
        colorscale=[[0, "#1D4ED8"], [0.5, "#F8FAFC"], [1, "#F59E0B"]],
        colorbar={"title": "Índice<br>sazonal"},
        hovertemplate="%{y}<br>%{x}<br>Índice sazonal: %{z:.1f}<extra></extra>",
    )
)
style_figure(fig, "Top 10 produtos com maior variação sazonal", height=570)
fig.update_xaxes(title="Mês")
fig.update_yaxes(title="Produto", autorange="reversed")
fig.show()

display(
    seasonality_strength.head(10).assign(
        seasonality_strength=lambda frame: frame["seasonality_strength"].round(2)
    ).rename(columns={"item": "Produto", "seasonality_strength": "Força sazonal (desvio-padrão do índice)"})
)

## 13. Síntese executiva

Os indicadores abaixo são calculados diretamente da base para manter as conclusões reproduzíveis.

In [ ]:
first_year = int(yearly_sales.iloc[0]["year"])
last_year = int(yearly_sales.iloc[-1]["year"])
total_growth = (yearly_sales.iloc[-1]["sales"] / yearly_sales.iloc[0]["sales"] - 1) * 100
peak_month = monthly_profile.loc[monthly_profile["sales"].idxmax()]
lowest_month = monthly_profile.loc[monthly_profile["sales"].idxmin()]
peak_weekday = weekday_profile.loc[weekday_profile["sales"].idxmax()]
top_store = store_sales.iloc[0]
top_item = item_sales.iloc[0]
weekend_average = daily_sales.loc[daily_sales["is_weekend"], "sales"].mean()
weekday_average = daily_sales.loc[~daily_sales["is_weekend"], "sales"].mean()
weekend_uplift = (weekend_average / weekday_average - 1) * 100
seasonal_amplitude = (peak_month["sales"] / lowest_month["sales"] - 1) * 100

insights = pd.DataFrame(
    {
        "Conclusão": [
            "Crescimento estrutural",
            "Pico mensal",
            "Amplitude sazonal",
            "Pico semanal",
            "Efeito do fim de semana",
            "Loja líder",
            "Produto líder",
            "Produto com maior variação sazonal",
        ],
        "Evidência": [
            f"As vendas cresceram {total_growth:.1f}% entre {first_year} e {last_year}.",
            f"{peak_month['month_name']} apresenta a maior média diária: {format_units(peak_month['sales'])}.",
            f"O mês mais forte supera o mais fraco em {seasonal_amplitude:.1f}%.",
            f"{peak_weekday['day_name']} registra a maior média diária: {format_units(peak_weekday['sales'])}.",
            f"Fins de semana vendem, em média, {weekend_uplift:.1f}% mais que dias úteis.",
            f"Loja {int(top_store['store'])}, com {format_units(top_store['sales'])} unidades no período.",
            f"Produto {int(top_item['item'])}, com {format_units(top_item['sales'])} unidades no período.",
            f"Produto {int(seasonality_strength.iloc[0]['item'])}, pelo maior desvio do índice mensal.",
        ],
        "Implicação para o negócio": [
            "Atualizar parâmetros de estoque para acompanhar o aumento do nível de demanda.",
            "Antecipar compras e capacidade operacional antes do período de pico.",
            "Usar estoques de segurança e metas diferentes ao longo do ano.",
            "Reforçar reposição e operação nos dias de maior fluxo.",
            "Planejar cobertura adicional para sábado e domingo.",
            "Investigar práticas da loja líder e diferenças de capacidade ou mercado.",
            "Priorizar disponibilidade dos itens líderes sem ignorar a cauda do portfólio.",
            "Aplicar parâmetros sazonais específicos a produtos com maior oscilação.",
        ],
    }
)
display(insights)

## Próxima etapa

A análise confirma tendência e padrões recorrentes relevantes para previsão. A Etapa 3 deverá criar lags e estatísticas móveis por combinação de `store` e `item`, sempre deslocando a variável `sales` antes das janelas para evitar vazamento de dados.